# ENSO longitude index change over time

## Imports

In [ ]:
import datetime
import matplotlib
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import src.XRO
import copy
import scipy.stats
import warnings
import calendar
import scipy.optimize

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False}, palette="colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Funcs

In [ ]:
def get_rolling_std(data, n=20):
    """
    Get standard deviation, computing over time and ensemble member. To increase
    sample size for variance estimate, compute over time window of 2n+1
    years, centered at given year.
    """

    ## do the computation
    kwargs = dict(fn=np.std, n=n, reduce_ensemble_dim=False)
    data_std = src.utils.get_rolling_fn_bymonth(data, **kwargs)

    ## unstack year and month
    data_std = src.utils.unstack_month_and_year(data_std)

    return data_std


def get_stats(x):
    """helper function to compute plotting bounds for experiment"""
    stats = x.quantile(q=[0.1, 0.5, 0.9], dim="member")
    return stats.rename({"quantile": "q"})


def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits


def get_fits_over_time_wrapper(
    data_rolling, model, by_member=False, fname=None, **fit_kwargs
):
    """wrapper function to handle saving/loading"""

    ## function to compute fits
    get_fits = lambda: get_fits_over_time(
        data_rolling, model=model, by_member=by_member, **fit_kwargs
    )

    ## if fname not specified, compute without loading/saving
    if fname is None:
        fits = get_fits()

    else:

        ## get full filepath
        fp = pathlib.Path(os.environ["SAVE_FP"], "fits_cesm", fname)

        ## try to load pre-computed
        if fp.is_file():
            print("Loading pre-computed")
            fits = xr.open_dataset(fp)

        ## otherwise, compute and save
        else:
            fits = get_fits()
            fits.to_netcdf(fp)

    return fits


def save(fig, fname, dpi=300):
    """save figure to file"""

    ## get save directory
    save_dir = pathlib.Path(os.environ["SAVE_FP"], "ch3-outline")

    ## get fname
    fname = save_dir / f"{fname}.pdf"

    fig.savefig(fname, dpi=dpi)


def get_RO_sigma(model, params, **simulation_kwargs):
    """Compute stats (e.g., standard deviation) for RO parameters over time"""

    output = model.simulate(fit_ds=params, **simulation_kwargs)

    return output.groupby("time.month").std()


def get_RO_sigma_over_time(model, params, **simulation_kwargs):
    """Compute stats over time"""

    ## take ensemble mean if necessary
    if "member" in params.dims:
        params = params.mean("member")

    ## empty list to hold result
    sigmas = []

    ## loop through years
    for y in tqdm.tqdm(params.year):

        ## specs for simulation
        kwargs = dict(
            simulation_kwargs,
            model=model,
            params=params.sel(year=y),
        )

        ## do the simulation
        sigmas.append(get_RO_sigma(**kwargs))

    ## put back in xarray
    sigmas = xr.concat(sigmas, dim=params.year)

    return sigmas


def plot_stats_comp(ax, list_of_stats, labels, n, colors=None):
    """plot comparison of variance over time"""

    if colors is None:
        colors = sns.color_palette()[: len(list_of_stats)]

    for stats, label, c in zip(list_of_stats, labels, colors):

        ## plot median
        mplot = ax.plot(stats.year, stats[n].sel(q=0.5), lw=2.5, label=label, c=c)

        ## plot lower/upper quantiles
        kwargs = dict(c=mplot[0].get_color(), lw=0.8)
        for q in stats.q:
            if q != 0.5:
                ax.plot(stats.year, stats[n].sel(q=q), **kwargs)

    ## label and set plotting specs
    ax.set_xlabel("Year")
    ax.set_ylabel(r"$\sigma_T$ ($^{\circ}$C)")
    ax.set_ylim([0.3, 1.7])
    ax.set_xticks([1870, 1975, 2080])
    ax.set_yticks([0.6, 1.2])

    return


def format_validation_plots(axs):
    """add formatting to CESM v. RO plot"""

    axs[1, 0].set_ylabel(r"$\sigma_h$($^{\circ}$C)")
    axs[0, 1].legend(prop=dict(size=8))
    for i in range(axs.shape[1]):
        ## remove ticks
        axs[0, i].set_xticks([])
        axs[0, i].set_xlabel(None)
        if i > 0:
            for ax in axs[:, i]:
                ax.set_yticks([])
                ax.set_ylabel(None)

    return

## Load data

### T, h

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True, load_h_cust=True, max_grad=True)

## omit first year (bc of NaN in h,hw vars)
Th = Th.sel(time=slice("1851", None))

## Load ELI
eli = xr.open_dataset(pathlib.Path(DATA_FP, "cesm/eli.nc"))
_, eli_anom = src.utils.separate_forced(eli)

## merge
Th = xr.merge([Th, eli_anom.sel(time=slice("1851", None))])

### preprocess

In [ ]:
## standardize (for convenience)
Th /= Th.std()

## get windowed data (used to estimate change in parameters over time)
Th_rolling = src.utils.get_windowed(Th, window_size=480, stride=120)

## compute grad
Th_rolling["dTdx"] = Th_rolling["T_3"] - Th_rolling["T_4"]

#### Scale by $\overline{h}$

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale.nc"),
)
# Th_rolling["h_w_Hbar-scaled"] = Th_rolling["h_w"] * hbar_scale
Th_rolling["h_w_Hbar-scaled"] = Th_rolling["h_w"] * hbar_scale

#### Scale ELI by $dT/dx$

In [ ]:
## load dT/dx data
dTdx = xr.open_dataarray(pathlib.Path(SAVE_FP, "cesm_dTdx.nc"))

## scale by initial value
dTdx_scale = dTdx / dTdx.isel(year=0).mean(["month"])

for v in list(Th_rolling):
    if "eli" in v:
        Th_rolling[f"{v}_scaled"] = Th_rolling[v].groupby("time.month") * dTdx_scale

## Fit RO

### Do fitting

In [ ]:
## specify filename for saving (if None, then don't save/load)
FNAME = "eli_05_scaled_h_w_Hbar_real"

## should we use forward differences to estimate operator?
IS_FORWARD = True

## specify variables
varnames = ["eli_05_scaled", "h_w_Hbar-scaled"]

## fit for each ensemble member differently
BY_MEMBER = False

## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=3, is_forward=IS_FORWARD)
fit_kwargs = dict(ac_mask_idx=None, maskNT=[])

## get fits
fits = get_fits_over_time_wrapper(
    Th_rolling[varnames],
    model=MODEL,
    by_member=BY_MEMBER,
    fname=FNAME,
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

### Stochastic integration

In [ ]:
## should we use noise covariance
# USE_NOISE_COV = False
USE_NOISE_COV = True

## simulation specs
simulation_kwargs = dict(
    nyear=40,
    ncopy=50,
    seed=1000,
    X0_ds=Th_rolling[varnames].isel(year=0, member=0, time=0),
    noise_type="white",
    use_noise_cov=USE_NOISE_COV,
    is_xi_stdac=True,
)

## compute with parameters estimated from all ensemble members
RO_sigma_over_time_v2 = get_RO_sigma_over_time(
    model=MODEL, params=fits, **simulation_kwargs
)

### Compute stats / validate

#### Stats

In [ ]:
## compute rolling std
# Th_std = get_rolling_std(Th, n=20)
Th_std = Th_rolling.groupby("time.month").std("time")

## compute percentage change in std
baseline = Th_std.isel(year=0).mean("member")
delta_Th_std = 100 * (Th_std - baseline) / baseline

#### Plot ELI variance

In [ ]:
## specify months to plot
PLOT_MONTHS = [2, 5, 8, 11]

fig, axs = plt.subplots(2, 4, figsize=(8, 4), layout="constrained")

for i, m in enumerate(PLOT_MONTHS):

    ## compute stats
    stats_mpi = get_stats(Th_std).sel(month=m)
    stats_ro_v2 = get_stats(RO_sigma_over_time_v2).sel(month=m)

    ## specify kwargs
    plot_kwargs = dict(
        list_of_stats=[stats_mpi, stats_ro_v2],
        labels=["CESM", "RO"],
        colors=["k", sns.color_palette()[1]],
    )

    ## plot comparison
    for j, ax in enumerate(axs[:, i]):
        plot_stats_comp(ax, n=varnames[j], **plot_kwargs)

    ## label
    axs[0, i].set_title(calendar.month_name[m])

## format all subplots
format_validation_plots(axs)

plt.show()

#### Plot transfer funcs

In [ ]:
V0 = "eli_05_scaled"
V1 = "T_34"

sel = lambda x: src.utils.sel_month(x, 2).stack(s=["member", "time"])
fig, axs = plt.subplots(1, 2, figsize=(6.5, 3), layout="constrained")
axs[0].scatter(
    sel(Th_rolling[V0].isel(year=0)),
    sel(Th_rolling[V1].isel(year=0)),
    s=3,
    alpha=0.5,
)

axs[1].scatter(
    sel(Th_rolling[V0].isel(year=-1)),
    sel(Th_rolling[V1].isel(year=-1)),
    s=3,
    alpha=0.5,
)

src.utils.set_lims(axs)

plt.show()

#### Get best fit

In [ ]:
def psi_(x, a, b, c):
    return c * np.exp(a * x) + b


## get data in np
xdata = sel(Th_rolling[V0].isel(year=-1)).values
ydata = sel(Th_rolling[V1].isel(year=-1)).values

## do fit
p, _ = scipy.optimize.curve_fit(f=psi_, xdata=xdata, ydata=ydata, p0=[-1, 1, -1])

## define transfer func
psi = lambda x: psi_(x, a=p[0], b=p[1], c=p[2])

In [ ]:
c = -1
a = -1.3
b = 1.5

yhat = c * np.exp(a * xdata) + b

fig, ax = plt.subplots(figsize=(3, 2.5))
ax.scatter(xdata, ydata, s=5)
ax.scatter(xdata, psi(xdata), c="k", s=3)
plt.show()